#### Billing and Calls Dataset Merger

Merge 1 — Renewal calls only: merges renewal_calls with billings on Co_Ref, filters to current renewal cycle within 14-45 day window, deduplicates, and saves `br_merged.csv`

In [0]:
import pandas as pd

renewal_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/renewal_calls.csv')
billings_df   = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/billings.csv', low_memory=False)

renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

renewal_calls = renewal_calls.dropna(subset=['Call_Date'])

billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])
billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

merged = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date'] > merged['Prev_Renewal_Date'])
        )
    )
]
print("Shape after cycle filter:", merged.shape)

merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Days_Before_Renewal'] >= 14) &
        (merged['Days_Before_Renewal'] <= 45)
    )
]
print("Shape after 14-45 day filter:", merged.shape)

merged = merged.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

counts = merged['Co_Ref'].value_counts()
valid_coref = counts[counts > 5].index
filtered = merged[merged['Co_Ref'].isin(valid_coref)]
filtered = filtered.sort_values(['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date'])
output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]
output.to_csv('./sample_filtered_renewal_coref.csv', index=False)
print("Saved debug file: ./sample_filtered_renewal_coref.csv")

merged.to_csv("/Workspace/Shared/DS_Miniproject/data/merged/br_merged.csv", index=False)
print("Saved final dataset: br_merged.csv")
print(merged)

Merge 2 — CC calls only: merges cc_calls with billings on Co_Ref, filters to current renewal cycle within 14-45 day window, deduplicates, and saves `bcc_merged.csv`

In [0]:
import pandas as pd

cc_calls    = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/cc_calls.csv')
billings_df = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/billings.csv', low_memory=False)

cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

cc_calls = cc_calls.dropna(subset=['Call_Date'])

billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])
billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

merged = billings_df.merge(cc_calls, on='Co_Ref', how='left')

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date'] > merged['Prev_Renewal_Date'])
        )
    )
]
print("Shape after cycle filter:", merged.shape)

merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Days_Before_Renewal'] >= 14) &
        (merged['Days_Before_Renewal'] <= 45)
    )
]
print("Shape after 14-45 day filter:", merged.shape)

merged = merged.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

counts = merged['Co_Ref'].value_counts()
valid_coref = counts[counts > 5].index
filtered = merged[merged['Co_Ref'].isin(valid_coref)]
filtered = filtered.sort_values(['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date'])
output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]
output.to_csv('./sample_filtered_cc_coref.csv', index=False)
print("Saved debug file: ./sample_filtered_cc_coref.csv")

merged.to_csv("/Workspace/Shared/DS_Miniproject/data/merged/bcc_merged.csv", index=False)
print("Saved final dataset: bcc_merged.csv")

Merge 3 — Full raw merge: sequentially merges renewal then CC calls with billings, applies cycle and 14-45 day filters for both, and saves `full_raw_billing_calls.csv`

In [0]:
import pandas as pd

renewal_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/renewal_calls.csv')
cc_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/cc_calls.csv')
billings_df = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/billings.csv', low_memory=False)

renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)
cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls = cc_calls.dropna(subset=['Call_Date'])

billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])
billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

merged = billings_df.merge(
    renewal_calls, on='Co_Ref', how='left', suffixes=('', '_renewal')
)

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date'] > merged['Prev_Renewal_Date'])
        )
    )
]

merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Days_Before_Renewal'] >= 14) &
        (merged['Days_Before_Renewal'] <= 45)
    )
]

merged = merged.merge(
    cc_calls, on='Co_Ref', how='left', suffixes=('', '_cc')
)

merged = merged[
    (merged['Call_Date_cc'].isna()) |
    (
        (merged['Call_Date_cc'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date_cc'] > merged['Prev_Renewal_Date'])
        )
    )
]

merged['cc_Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date_cc']
).dt.days

merged = merged[
    (merged['Call_Date_cc'].isna()) |
    (
        (merged['cc_Days_Before_Renewal'] >= 14) &
        (merged['cc_Days_Before_Renewal'] <= 45)
    )
]

print("Final dataset shape:", merged.shape)
print(merged.head())

merged.to_csv("/Workspace/Shared/DS_Miniproject/data/merged/full_raw_billing_calls.csv", index=False)
print("Saved: full_raw_billing_calls.csv")

Merge 4 — Aggregated call counts: processes renewal and CC calls separately, aggregates to per-cycle counts, merges back to billings, and saves `final_billing_calls.csv` with sample debug output

In [0]:
import pandas as pd

renewal_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/renewal_calls.csv')
cc_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/cc_calls.csv')
billings_df = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/billings.csv', low_memory=False)

renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)
cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])
billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

br = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

br = br[
    (br['Call_Date'].isna()) |
    (
        (br['Call_Date'] <= br['Prospect_Renewal_Date']) &
        (
            (br['Prev_Renewal_Date'].isna()) |
            (br['Call_Date'] > br['Prev_Renewal_Date'])
        )
    )
]

br['Days_Before_Renewal'] = (
    br['Prospect_Renewal_Date'] - br['Call_Date']
).dt.days

br = br[
    (br['Call_Date'].isna()) |
    (
        (br['Days_Before_Renewal'] >= 14) &
        (br['Days_Before_Renewal'] <= 45)
    )
]

br = br.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

br_agg = br.groupby(['Co_Ref', 'Prospect_Renewal_Date']).agg(
    renewal_call_count_14_45=('Call_Date', 'count')
).reset_index()

bcc = billings_df.merge(cc_calls, on='Co_Ref', how='left')

bcc = bcc[
    (bcc['Call_Date'].isna()) |
    (
        (bcc['Call_Date'] <= bcc['Prospect_Renewal_Date']) &
        (
            (bcc['Prev_Renewal_Date'].isna()) |
            (bcc['Call_Date'] > bcc['Prev_Renewal_Date'])
        )
    )
]

bcc['Days_Before_Renewal'] = (
    bcc['Prospect_Renewal_Date'] - bcc['Call_Date']
).dt.days

bcc = bcc[
    (bcc['Call_Date'].isna()) |
    (
        (bcc['Days_Before_Renewal'] >= 14) &
        (bcc['Days_Before_Renewal'] <= 45)
    )
]

bcc = bcc.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

bcc_agg = bcc.groupby(['Co_Ref', 'Prospect_Renewal_Date']).agg(
    cc_call_count_14_45=('Call_Date', 'count')
).reset_index()

final_df = billings_df.merge(
    br_agg, on=['Co_Ref', 'Prospect_Renewal_Date'], how='left'
)

final_df = final_df.merge(
    bcc_agg, on=['Co_Ref', 'Prospect_Renewal_Date'], how='left'
)

final_df['renewal_call_count_14_45'] = final_df['renewal_call_count_14_45'].fillna(0)
final_df['cc_call_count_14_45']      = final_df['cc_call_count_14_45'].fillna(0)

print("Final dataset shape:", final_df.shape)
print(final_df.head())

final_df.to_csv("/Workspace/Shared/DS_Miniproject/data/merged/final_billing_calls.csv", index=False)
print("Saved final dataset: final_billing_calls.csv")

counts = final_df.groupby('Co_Ref')[
    ['renewal_call_count_14_45', 'cc_call_count_14_45']
].sum()
counts['total'] = counts.sum(axis=1)
valid_coref = counts[counts['total'] >= 0].index

br_sample = br[br['Co_Ref'].isin(valid_coref)][
    ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
].rename(columns={'Call_Date': 'Renewal_Call_Date'})

bcc_sample = bcc[bcc['Co_Ref'].isin(valid_coref)][
    ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
].rename(columns={'Call_Date': 'CC_Call_Date'})

sample_calls = pd.merge(
    br_sample, bcc_sample,
    on=['Co_Ref', 'Prospect_Renewal_Date'], how='outer'
)

sample_calls = sample_calls.sort_values(
    ['Co_Ref', 'Prospect_Renewal_Date', 'Renewal_Call_Date', 'CC_Call_Date']
)

print("\nSample with Call Dates:")
print(sample_calls.head(30))

sample_calls.to_csv('./sample_call_dates.csv', index=False)
print("Saved: ./sample_call_dates.csv")

Merge 5 — Latest call per renewal cycle: takes only the most recent renewal and CC call within each 14-45 day window, preserves all call columns, and saves `final_latest_calls_fullcols.csv`

In [0]:
import pandas as pd

renewal_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/renewal_calls.csv')
cc_calls = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/cc_calls.csv')
billings_df = pd.read_csv('/Workspace/Shared/DS_Miniproject/data/processed/billings.csv', low_memory=False)

print("Billing Shape:", billings_df.shape)
print("CC Calls Shape:", cc_calls.shape)
print("Renewal Calls Shape:", renewal_calls.shape)

renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)
cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)
billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])
billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

br = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prev_Renewal_Date']].merge(
    renewal_calls, on='Co_Ref', how='left'
)

br = br[
    (br['Call_Date'] <= br['Prospect_Renewal_Date']) &
    (
        (br['Prev_Renewal_Date'].isna()) |
        (br['Call_Date'] > br['Prev_Renewal_Date'])
    )
]

br['Days_Before_Renewal'] = (
    br['Prospect_Renewal_Date'] - br['Call_Date']
).dt.days

br = br[
    (br['Days_Before_Renewal'] >= 14) &
    (br['Days_Before_Renewal'] <= 45)
]

br_latest = br.sort_values('Call_Date').groupby(
    ['Co_Ref', 'Prospect_Renewal_Date']
).tail(1)

renewal_cols = [col for col in renewal_calls.columns if col != 'Co_Ref']
br_latest = br_latest[['Co_Ref', 'Prospect_Renewal_Date'] + renewal_cols]
br_latest = br_latest.rename(
    columns={col: col + '_renewal' for col in renewal_cols if col != 'Co_Ref'}
)

bcc = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prev_Renewal_Date']].merge(
    cc_calls, on='Co_Ref', how='left'
)

bcc = bcc[
    (bcc['Call_Date'] <= bcc['Prospect_Renewal_Date']) &
    (
        (bcc['Prev_Renewal_Date'].isna()) |
        (bcc['Call_Date'] > bcc['Prev_Renewal_Date'])
    )
]

bcc['Days_Before_Renewal'] = (
    bcc['Prospect_Renewal_Date'] - bcc['Call_Date']
).dt.days

bcc = bcc[
    (bcc['Days_Before_Renewal'] >= 14) &
    (bcc['Days_Before_Renewal'] <= 45)
]

bcc_latest = bcc.sort_values('Call_Date').groupby(
    ['Co_Ref', 'Prospect_Renewal_Date']
).tail(1)

cc_cols = [col for col in cc_calls.columns if col != 'Co_Ref']
bcc_latest = bcc_latest[['Co_Ref', 'Prospect_Renewal_Date'] + cc_cols]
bcc_latest = bcc_latest.rename(
    columns={col: col + '_cc' for col in cc_cols if col != 'Co_Ref'}
)

final_df = billings_df.merge(
    br_latest, on=['Co_Ref', 'Prospect_Renewal_Date'], how='left'
)
final_df = final_df.merge(
    bcc_latest, on=['Co_Ref', 'Prospect_Renewal_Date'], how='left'
)

print("Final dataset shape:", final_df.shape)
print(final_df.head())

final_df.to_csv("/Workspace/Shared/DS_Miniproject/data/merged/final_latest_calls_fullcols.csv", index=False)
print("Saved: final_latest_calls_fullcols.csv")